# BeeHappy — Data Foundation

Three tables of real beehive sensor data go in. One model-ready dataframe comes out.

The data is not on your disk. It lives in a Postgres database, and the first thing you
do is go and get it.

Nobody has cleaned this for you and **you are not told what is wrong with it**. Finding
that out is the work. One habit matters more than any other here:

> **Check your row count after every join and every reshape.**
> If it changes and you cannot say exactly why, stop and find out.

Roughly 4 hours. Exercises build on each other, so do them in order.

## The target

One row per hive per hour, with these columns. `workshop/validate_features.py`
enforces exactly this, so match the names.

| Column | Meaning |
|---|---|
| `beehive_id`, `hive_name` | which hive |
| `hour` | the hour, UTC |
| `brood_temp_c1/c2/c3` | brood chamber, three probes |
| `food_temp`, `food_humidity` | food chamber |
| `outside_temp`, `outside_humidity`, `outside_wind_speed`, `outside_wind_dir`, `outside_uv_index`, `outside_pressure_pa`, `outside_light`, `outside_rain` | ambient |
| `hour_local`, `month`, `dayofweek` | calendar, **Europe/Berlin** |

Write it to `workshop/out/features_hourly.parquet`, then run:

```bash
uv run python workshop/validate_features.py workshop/out/features_hourly.parquet
```

In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

REPO = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
OUT = REPO / "workshop" / "out"
RAW = OUT / "raw"          # your extract lands here
RAW.mkdir(parents=True, exist_ok=True)

# The connection string comes from .env, which is gitignored. The role is
# read-only: you cannot damage the database, only wait a long time for a query
# you did not mean to run.
load_dotenv(REPO / ".env")
DATABASE_URL = os.environ["DATABASE_URL"]
print("database:", DATABASE_URL.rsplit("@", 1)[-1])
print("extract ->", RAW)

---
## Exercise 0 — Extract (30 min)

The source is a Postgres database with three tables:

| Table | What it is |
|---|---|
| `beehives` | the hives |
| `sensors` | the devices, and which hive each belongs to |
| `data` | every reading, one row per measurement |

**Do this:**
1. Open **one** connection using `DATABASE_URL`.
2. Pull `beehives` and `sensors` whole. They are tiny.
3. Pull the **last three months** of `data`. It is a large table and most of it is
   older than that, so filter in SQL — not in pandas after the fact.
4. Persist all three to `workshop/out/raw/` in whatever format you prefer: Parquet,
   CSV, a local SQLite file, your choice. Be ready to say why you chose it.
5. Close the connection, then **read your saved copy back into dataframes** and work
   from those for the rest of the day.

**Deliverable:** `beehives`, `sensors` and `data` in memory, loaded from disk, with row
counts and the exact period you pulled printed out.

*You should hit the database once, today. If you find yourself re-running a query
later to get a dataframe back, the extract step is not finished.*

*Mind your format. `ts` is a timestamp and `value` is a float — CSV will hand both back
to you as strings unless you tell it otherwise.*

In [ ]:
# TODO: extract once, persist, read back.
#
# import psycopg
#
# SQL = {
#     "beehives": "SELECT * FROM beehives",
#     "sensors":  "SELECT * FROM sensors",
#     "data":     "SELECT * FROM data WHERE ts >= ...",   # last three months
# }
#
# `ts` is a naive timestamp column. Before you write that WHERE clause,
# work out what `now()` is measured against and whether it matches.
#
# with psycopg.connect(DATABASE_URL, connect_timeout=30) as conn:
#     with conn.cursor() as cur:
#         cur.execute(...)
#         # cur.fetchall() gives tuples; [c.name for c in cur.description] the columns
#         frame = pd.DataFrame(...)
#
# ... write each frame under RAW ...
#
# Then read back, and print what you actually got:
# data = pd.read_parquet(RAW / "data.parquet")
# print(len(data), data["ts"].min(), "->", data["ts"].max())

---
## Exercise 1 — Profile (30 min)

Before touching anything, find out what you have.

**Answer these:**
1. What is one row of `data`? What makes a row unique?
2. How many measurements exist, and which devices produce which?
3. What period is covered, and are there NULLs?
4. How do `sensors` and `beehives` relate to `data`?

**Deliverable:** a short written profile, plus at least one thing that surprises you.

*Hint: `data` is **long** — one row per measurement, not per device reading.*

In [ ]:
# TODO: profile the three tables.
#   data.head(), data.dtypes, data["measurement_unit"].value_counts()
#   sensors  -- how many devices, of which types, on which hives?
#   Does data have NULLs? Does that mean nothing is missing?

---
## Exercise 2 — Deduplicate (30 min)

**Answer these:**
1. Which combination of columns *should* uniquely identify a row?
2. Does it? By how much?
3. Where duplicates exist, do they agree on `value`? This decides whether you can
   simply drop them or have to arbitrate between conflicting readings.

**Deliverable:** `data` at one row per real measurement, and a number for how much
you removed.

*`reading_id` is a database row id. Use it to prove what you find is real and not an
artifact of your own query.*

In [ ]:
# TODO: find the true grain, quantify the redundancy, check for conflicts, dedup.
# KEY = [...]

---
## Exercise 3 — Reshape and join (60 min)

Get from long to wide, then attach context: which hive a reading belongs to, and what
the weather was doing.

**Answer these:**
1. Pivot so each measurement is a column. What is the row grain now?
2. Ambient readings are not a property of a hive. How should they attach?
3. **After every join, did the row count change? Is that change correct?**

**Deliverable:** a hive-level frame and a weather frame, both with a grain you can state.

*Take question 3 seriously. There is something in `sensors` that makes a careless join
here silently wrong, and it will not raise an error.*

In [ ]:
# TODO: map sensor_type -> a readable role, split hive vs ambient, pivot to wide.
# ROLE = {
#     "LoRaWAN Dragino-S31-LB": "food_chamber",
#     "LoRaWAN Dragino-D23-LB": "brood_chamber",
#     "LoRaWAN SenseCAP-S2120": "weather",
# }
# Print len(...) before and after every merge.

---
## Exercise 4 — Align time and handle gaps (60 min)

Devices report on their own schedules and their timestamps never line up, so nothing
can be joined on an exact time. Put everything on one clock.

**Answer these:**
1. Pick a resolution and justify it — how often does each device actually report?
2. Build a **complete** index of every hive and every hour in the period. Compare it
   to what you have. How many hive-hours are missing?
3. What are the gaps? A few minutes, or days? Look at the distribution before choosing
   a treatment.
4. Interpolate, forward-fill, or leave NaN? Different answers for a 1-hour gap and a
   5-day outage are fine — but write down the rule.

**Deliverable:** one row per hive-hour, with a stated gap policy.

*This is where the missing data actually is. There are no NULLs in this dataset — rows
for hours a device did not report simply do not exist. Only a complete index reveals them.*

*Build the index from the window you extracted, not from what one device happens to
cover. If a hive was silent for the last week of your three months, its rows end early
and the grid should not — that silence is the thing you are trying to see.*

In [ ]:
# TODO: floor to the hour, aggregate, reindex onto the full grid, measure the gaps.
# hours = pd.date_range(..., ..., freq="h")
# grid  = pd.MultiIndex.from_product([...], names=["beehive_id", "hour"])

---
## Exercise 5 — Validate and document (30 min)

**Do this:**
1. Add the calendar columns. Careful: `ts` is UTC and the hive is in Germany, so
   "3am" in the data is not 3am at the hive. Germany changes its clocks, so the offset
   is not a constant you can hardcode — check whether your three months cross a change.
2. Rename to the target schema above.
3. Write assertions *before* running the validator — grain, row count, ranges.
4. Write `DATA_DICTIONARY.md`: every column, its unit, and **every decision you made** —
   what window you extracted, what you dropped, what you filled, what you left empty
   and why.
5. Save and validate.

**Deliverable:** `features_hourly.parquet` + `DATA_DICTIONARY.md` passing the validator.

In [ ]:
# TODO: calendar features in Europe/Berlin, rename, assert, save.
# features.to_parquet(OUT / "features_hourly.parquet", index=False)

```bash
uv run python workshop/validate_features.py workshop/out/features_hourly.parquet
```

Green is not the goal. A table that passes but whose gap policy you cannot defend is
worse than one that fails honestly.